In [ ]:
import os
import sys
import shutil
import ctypes

def is_admin():
    """檢查目前是否擁有系統管理員權限"""
    try:
        return ctypes.windll.shell32.IsUserAnAdmin()
    except:
        return False

def move_app_with_link(src, dst):
    """
    src: C槽原本的軟體資料夾路徑 (例如 "C:\\Program Files\\TargetApp")
    dst: D槽目標的軟體資料夾路徑 (例如 "D:\\Games\\TargetApp")
    """
    # 1. 安全檢查
    if not os.path.exists(src):
        print(f"❌ 錯誤：找不到來源資料夾 {src}，請確認路徑是否正確。")
        return
    
    if os.path.exists(dst):
        print(f"❌ 錯誤：目標路徑 {dst} 已經存在檔案或資料夾，請先刪除或更名。")
        return

    # 確保 D 槽的上一層資料夾存在
    dst_parent = os.path.dirname(dst)
    if not os.path.exists(dst_parent):
        os.makedirs(dst_parent)

    try:
        # 2. 開始搬移檔案
        print(f"📂 正在將檔案從 C 槽複製到 D 槽...")
        print(f"來源: {src}")
        print(f"目標: {dst}")
        
        # 使用 shutil.move 搬移整個資料夾
        shutil.move(src, dst)
        print("✅ 檔案搬移成功！")

        # 3. 建立軟連結（傳送門）
        print("🔗 正在建立系統軟連結...")
        # 組合 Windows 的 mklink /J 指令
        cmd = f'mklink /J "{src}" "{dst}"'
        
        # 執行指令
        result = os.system(cmd)
        
        if result == 0:
            print("\n🎉 大功告成！軟體已成功轉移至 D 槽，且系統傳送門建立完畢。")
            print("您現在可以照常啟動該軟體了。")
        else:
            print("\n❌ 軟連結建立失敗，請手動確認 C 槽原路徑是否被佔用。")

    except Exception as e:
        print(f"\n💥 發生錯誤: {e}")
        print("請確保該軟體已經「完全關閉」，背景沒有任何相關進程在執行。")

if __name__ == "__main__":
    # 檢查權限，若無管理員權限則自動重新以管理員身分呼叫自己
    if not is_admin():
        print("🔰 正在請求系統管理員權限...")
        ctypes.windll.shell32.ShellExecuteW(None, "runas", sys.executable, " ".join(sys.argv), None, 1)
    else:
        print("==============================================")
        print("         Windows 應用程式安全轉移工具          ")
        print("==============================================")
        
        # 🛠️ 在這裡輸入你要搬移的實際路徑（路徑中的斜線請用雙斜線 \\ 或正斜線 /）
        old_path = r"C:\Program Files\ExampleApp"  # 👈 換成你的 C 槽軟體資料夾
        new_path = r"D:\Apps\ExampleApp"          # 👈 換成你的 D 槽目標資料夾
        
        # 執行搬移
        move_app_with_link(old_path, new_path)
        
        # 防止視窗執行完直接閃退
        input("\n按任意鍵結束程式...")
